# 06 - 热方程正问题

方程：`u_t = alpha u_xx`，`alpha=0.2`。

初值 `u(x,0)=sin(pi*x)`，边界 `u(0,t)=u(1,t)=0`。

解析解：`sin(pi*x)*exp(-alpha*pi^2*t)`。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(6)
device = "cuda" if torch.cuda.is_available() else "cpu"
alpha = 0.2
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, xt):
        return self.net(xt)

model = MLP().to(device)

def exact_solution(x, t):
    return torch.sin(math.pi * x) * torch.exp(-alpha * math.pi**2 * t)

In [ ]:
def residual(x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)
    u = model(torch.cat([x, t], dim=1))
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t - alpha * u_xx

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(8000):
    x_f = torch.rand(512, 1, device=device)
    t_f = torch.rand(512, 1, device=device)
    loss_phys = torch.mean(residual(x_f, t_f)**2)

    x_i = torch.rand(128, 1, device=device)
    t_i = torch.zeros_like(x_i)
    loss_ic = torch.mean((model(torch.cat([x_i, t_i], dim=1)) - exact_solution(x_i, t_i))**2)

    t_b = torch.rand(128, 1, device=device)
    x_b = torch.cat([torch.zeros(64, 1, device=device), torch.ones(64, 1, device=device)], dim=0)
    loss_bc = torch.mean(model(torch.cat([x_b, t_b], dim=1))**2)

    loss = loss_phys + 10 * loss_ic + 10 * loss_bc
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append([loss.item(), loss_phys.item(), loss_ic.item(), loss_bc.item()])
    if epoch % 1600 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.3e}")

In [ ]:
x = torch.linspace(0, 1, 300, device=device).reshape(-1, 1)
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
for tv in [0.0, 0.2, 0.5, 1.0]:
    t = torch.full_like(x, tv)
    with torch.no_grad():
        pred = model(torch.cat([x, t], dim=1))
        exact = exact_solution(x, t)
    plt.plot(x.cpu(), exact.cpu(), label=f"exact t={tv}")
    plt.plot(x.cpu(), pred.cpu(), "--")
plt.legend(fontsize=8)
plt.title("Heat equation")

plt.subplot(1, 2, 2)
plt.semilogy(torch.tensor(history)[:, 0])
plt.title("Total loss")
plt.tight_layout()
plt.show()